# 04 · Memory Chatbot

**목표**: LLM이 이전 호출을 스스로 기억하지 않는다는 점을 확인하고, 대화 history를 매번 전달해 문맥을 유지합니다.

챗봇의 기억은 이전 user와 assistant 메시지를 다음 요청에 함께 보내는 것입니다.

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #2e7d32; background: #e8f5e9; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #2e7d32; letter-spacing: .02em;">🖥️  UI 실행</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">UI: Memory Chatbot</h2>
<p style="margin: 7px 0;">이 UI는 UI에서 바로 구성한 bot instance를 그대로 사용합니다. Reset 버튼은 bot history를 비우므로, UI의 Clear와 구분됩니다.</p>
  </div>
</div>


In [ ]:
from pathlib import Path
import sys
from typing import List
import gradio as gr

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
from lab_common.config import load_settings
from lab_common.ai_core import apply_ai_core_environment, orchestration_url
from gen_ai_hub.orchestration_v2.models.llm_model_details import LLMModelDetails
from gen_ai_hub.orchestration_v2.models.message import ChatMessage, SystemMessage, UserMessage
from gen_ai_hub.orchestration_v2.models.template import Template, PromptTemplatingModuleConfig
from gen_ai_hub.orchestration_v2.models.config import ModuleConfig, OrchestrationConfig
from gen_ai_hub.orchestration_v2.service import OrchestrationService

class UIMemoryBot:
    def __init__(self):
        settings = load_settings()
        apply_ai_core_environment(settings)
        template = Template(template=[
            SystemMessage(content='You are a helpful chatbot assistant.'),
            UserMessage(content='{{?user_query}}'),
        ])
        prompt = PromptTemplatingModuleConfig(
            prompt=template, model=LLMModelDetails(name=settings.llm_model, params={'max_tokens': 500})
        )
        self.service = OrchestrationService(api_url=orchestration_url(settings))
        self.config = OrchestrationConfig(modules=ModuleConfig(prompt_templating=prompt))
        self.history: List[ChatMessage] = []

    def chat(self, message):
        response = self.service.run(config=self.config, placeholder_values={'user_query': message}, history=self.history)
        answer = response.final_result.choices[0].message
        self.history = response.intermediate_results.templating
        self.history.append(answer)
        return answer.content

    def reset(self):
        self.history = []

ui_bot = None

def memory_reply(message, history):
    global ui_bot
    try:
        if ui_bot is None:
            ui_bot = UIMemoryBot()
        answer = ui_bot.chat(message)
        history = list(history or []) + [{'role': 'user', 'content': message}, {'role': 'assistant', 'content': answer}]
        return '', history, '응답에 이전 대화가 전달되었습니다.'
    except Exception as error:
        return message, history, f'챗봇 호출 실패: {type(error).__name__}: {error}'

def reset_memory():
    global ui_bot
    if ui_bot is not None:
        ui_bot.reset()
    return [], '대화 화면과 bot history를 초기화했습니다.'

with gr.Blocks() as memory_ui:
    gr.Markdown('## Memory Chatbot Playground')
    chatbot = gr.Chatbot(type='messages', label='Conversation')
    message = gr.Textbox(label='Message', placeholder='먼저 이름을 말한 뒤, 다음 메시지에서 이름을 물어보세요.')
    with gr.Row():
        send = gr.Button('Send', variant='primary')
        reset = gr.Button('Reset memory')
    status = gr.Textbox(label='Status', interactive=False)
    send.click(memory_reply, inputs=[message, chatbot], outputs=[message, chatbot, status])
    message.submit(memory_reply, inputs=[message, chatbot], outputs=[message, chatbot, status])
    reset.click(reset_memory, outputs=[chatbot, status])
memory_ui.launch(inline=True, share=False, show_error=True, server_name="127.0.0.1", height=900)


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1a73e8; background: #e8f0fe; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1a73e8; letter-spacing: .02em;">✅  체크포인트</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">체크포인트</h2>
<ul style="margin: 7px 0 0; padding-left: 1.3em;">
<li>UI에서 한 번 실행해 결과를 확인한다.</li>
<li>결과가 기대와 다르면 환경 설정과 모델/저장소 이름을 먼저 점검한다.</li>
<li>아래 코드는 같은 기능의 구성 요소를 자세히 살펴보는 선택 심화 자료다.</li>
</ul>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #5f6368; background: #f1f3f4; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #5f6368; letter-spacing: .02em;">🔎  선택 심화</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">선택: 코드로 같은 작업 보기</h2>
<p style="margin: 7px 0;">아래 셀은 UI가 사용하는 구성과 동작을 단계별로 확인하는 자료입니다. 기본 실습에서는 실행하지 않아도 됩니다.</p>
  </div>
</div>


<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">1. ChatBot과 history 관리</h2>
<p style="margin: 7px 0;">history=self.history가 이전 대화를 모델 호출에 전달합니다. 응답 뒤에는 이번 assistant 메시지를 history에 추가해 다음 턴에 사용합니다. reset은 history를 비워 새 대화를 시작합니다.</p>
  </div>
</div>


In [ ]:
from pathlib import Path
import sys
project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'shared').exists())
sys.path.insert(0, str(project_root / 'shared'))
from lab_common.config import load_settings
from lab_common.ai_core import api_headers, apply_ai_core_environment, orchestration_url

settings = load_settings()
apply_ai_core_environment(settings)
ORCH = orchestration_url(settings)

from typing import List

from gen_ai_hub.orchestration_v2.models.llm_model_details import LLMModelDetails
from gen_ai_hub.orchestration_v2.models.message import ChatMessage, SystemMessage, UserMessage
from gen_ai_hub.orchestration_v2.models.template import Template, PromptTemplatingModuleConfig
from gen_ai_hub.orchestration_v2.models.config import ModuleConfig, OrchestrationConfig
from gen_ai_hub.orchestration_v2.service import OrchestrationService

print("=" * 56)
print("Step 6 · 메모리 챗봇")
print("=" * 56)


class ChatBot:
    def __init__(self, service: OrchestrationService):
        self.service = service
        self.template = Template(template=[
            SystemMessage(content="You are a helpful chatbot assistant."),
            UserMessage(content="{{{?user_query}}}"),
        ])
        self.llm = LLMModelDetails(name=settings.llm_model, params={"max_tokens": 500})
        self.prompt_template = PromptTemplatingModuleConfig(prompt=self.template, model=self.llm)
        self.config = OrchestrationConfig(modules=ModuleConfig(prompt_templating=self.prompt_template))
        self.history: List[ChatMessage] = []

    def chat(self, user_input):
        response = self.service.run(
            config=self.config,
            placeholder_values={"user_query": user_input},
            history=self.history,
        )
        message = response.final_result.choices[0].message
        self.history = response.intermediate_results.templating
        self.history.append(message)
        return message.content

    def reset(self):
        self.history = []


service = OrchestrationService(api_url=ORCH)
bot = ChatBot(service=service)

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">2. 기억되는 대화</h2>
<p style="margin: 7px 0;">이름을 소개한 뒤 다른 질문을 하고 마지막에 이름을 물어봅니다. history가 유지되면 중간 질문이 있어도 첫 번째 정보를 참조할 수 있습니다.</p>
  </div>
</div>


In [ ]:
print("\n[1] 대화 (기억 테스트)")
print("  U: Hi, my name is Jin.")
print("  A:", bot.chat("Hi, my name is Jin."))
print("  U: Do you like SAP CodeJams?")
print("  A:", bot.chat("Do you like SAP CodeJams?"))
print("  U: What is my name?")
ans = bot.chat("What is my name?")
print("  A:", ans)
print("  → 이름 기억? ", "Jin" in ans)

<div style="margin: 18px 0 12px; padding: 13px 15px; border-left: 4px solid #1976d2; background: #eef6ff; border-radius: 6px; color: #202124; box-shadow: 0 1px 2px rgba(60,64,67,.12);">
  <div style="font-size: .82em; font-weight: 700; color: #1976d2; letter-spacing: .02em;">📘  개념 · 실습 단계</div>
  <div style="margin-top: 8px; font-size: .94em; line-height: 1.6;">
<h2 style="margin: 0 0 8px; font-size: 1.05em; line-height: 1.35; color: #202124;">3. reset 후 비교</h2>
<p style="margin: 7px 0;">reset 뒤 같은 질문을 반복합니다. 이전 메시지를 전달하지 않으므로 이름을 알 수 없어야 합니다. 장기 대화에서는 context window와 개인정보 보관 정책도 함께 설계해야 합니다.</p>
  </div>
</div>


In [ ]:
print("\n[2] reset 후 (기억 못 함 테스트)")
bot.reset()
ans2 = bot.chat("What is my name?")
print("  A:", ans2)
print("  → 리셋 후 이름 모름? ", "Jin" not in ans2)

print("\n✅ Step 6 완료 — history 로 맥락 기억, reset 으로 초기화.")